# 動画からシャトルを検知する — Gemini Robotics ER 2

バドミントンの動画（Google Drive に置いたもの）から、**シャトルがどこにあるか**を ER 2 に指させます。

やること
1. 動画からコマを取り出す（1秒に N 枚）
2. 1コマずつ ER 2 に「シャトルを指して」と聞く（見えなければ `null`）
3. 返ってきた点をコマに描いて並べる
4. 描いたコマをつなげて動画にする

前提: 左の 🔑（シークレット）に `GEMINI_API_KEY` を登録し、「ノートブックからのアクセス」を ON にしておく。


## 1. セットアップ

In [ ]:
%pip install -U -q "google-genai>=2.9.0" pydantic japanize-matplotlib

In [ ]:
from google.colab import userdata
from google import genai

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("ERROR: 左の 🔑 に GEMINI_API_KEY を登録し、ノートブックからのアクセスを ON にしてください")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-robotics-er-2-preview"

response = client.interactions.create(model=MODEL_ID, input="Hello Physical World?")
print(response.output_text)


In [ ]:
import base64
import json
import time
from io import BytesIO
from typing import List, Optional

import cv2
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
from PIL import Image, ImageDraw
from pydantic import BaseModel, Field
from IPython.display import HTML, display


def pil_to_b64(img, max_w=800):
    """PIL 画像を幅 max_w に縮めて base64 (PNG) にする。座標は 0〜1000 の正規化なので縮めても対応は変わらない。"""
    if img.size[0] > max_w:
        img = img.resize((max_w, int(max_w * img.size[1] / img.size[0])), Image.Resampling.LANCZOS)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


## 2. 動画を Google Drive から読む

動画を `マイドライブ` に置いて、`VIDEO_PATH` をそのファイル名に変えます（ASCII 名が安全）。
コマ数は **`SAMPLE_FPS × 秒数`** だけ ER 2 を呼ぶので、まず短い区間で試してください。


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

VIDEO_PATH = '/content/drive/MyDrive/badminton.mp4'   # ← 自分の動画に変える
START_SEC  = 0.0     # この秒から
DURATION   = 5.0     # この秒数ぶん
SAMPLE_FPS = 2       # 1秒あたり何コマ聞くか（呼び出し回数 = DURATION × SAMPLE_FPS）

assert os.path.exists(VIDEO_PATH), f"見つかりません: {VIDEO_PATH}"

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{w}x{h}, {fps:.1f} fps, {n_total / fps:.1f} 秒")

frames = []   # (秒, PIL画像)
t = START_SEC
while t < START_SEC + DURATION and t * fps < n_total:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(round(t * fps)))
    ok, bgr = cap.read()
    if not ok:
        break
    frames.append((t, Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))))
    t += 1.0 / SAMPLE_FPS
cap.release()
print(f"{len(frames)} コマ取り出しました（ER 2 を {len(frames)} 回呼びます）")

plt.figure(figsize=(8, 4.5)); plt.imshow(frames[0][1]); plt.axis("off"); plt.title(f"{frames[0][0]:.1f}s"); plt.show()


## 3. 1コマずつ ER 2 に「シャトルを指して」と聞く

シャトルは小さくて速いので、**見えないコマでは `null` を返させる**のが要点です。
「必ず1つ指して」と頼むと、無いところに点を置きます。


In [ ]:
class ShuttleAnswer(BaseModel):
    point: Optional[List[int]] = Field(
        default=None,
        description="シャトルの位置 [y, x]（0〜1000 に正規化）。見えなければ null")
    visible: bool = Field(description="シャトルがこのコマに写っているか")
    note: str = Field(description="判断の根拠を短く（日本語）")


PROMPT = (
    "バドミントンの試合の1コマです。シャトル（羽根）の位置を1点で指してください。"
    "シャトルが写っていない、または判別できない場合は point を null、visible を false にしてください。"
    "ラケットや白い線をシャトルと見間違えないこと。"
)


def ask_shuttle(img):
    r = client.interactions.create(
        model=MODEL_ID,
        input=[{"type": "user_input", "content": [
            {"type": "image", "data": pil_to_b64(img), "mime_type": "image/png"},
            {"type": "text", "text": PROMPT},
        ]}],
        generation_config={"thinking_level": "low"},
        response_format={"type": "text", "mime_type": "application/json",
                         "schema": ShuttleAnswer.model_json_schema()},
    )
    return ShuttleAnswer.model_validate_json(r.output_text)


results = []
for i, (sec, img) in enumerate(frames):
    ans = ask_shuttle(img)
    results.append((sec, ans))
    print(f"[{i+1}/{len(frames)}] {sec:5.1f}s  visible={ans.visible}  point={ans.point}  {ans.note}")
    time.sleep(0.5)


## 4. 返ってきた点をコマに描いて確かめる

点が本当にシャトルの上に乗っているかは、**目で確かめる**しかありません。
外れているコマがあれば、そのコマだけ聞き直す・`thinking_level` を上げる・コマを拡大して渡す、などを試します。


In [ ]:
def draw_point(img, ans, sec):
    out = img.copy()
    d = ImageDraw.Draw(out)
    W, H = out.size
    if ans.visible and ans.point:
        y, x = ans.point
        px, py = x / 1000 * W, y / 1000 * H
        r = max(8, W // 80)
        d.ellipse([px - r, py - r, px + r, py + r], outline=(255, 40, 40), width=max(3, W // 300))
        d.line([px - 2 * r, py, px + 2 * r, py], fill=(255, 40, 40), width=2)
        d.line([px, py - 2 * r, px, py + 2 * r], fill=(255, 40, 40), width=2)
        label = f"{sec:.1f}s"
    else:
        label = f"{sec:.1f}s  (not visible)"
    d.rectangle([0, 0, 260, 34], fill=(0, 0, 0))
    d.text((8, 8), label, fill=(255, 255, 255))
    return out


annotated = [draw_point(img, ans, sec) for (sec, img), (_, ans) in zip(frames, results)]

cols = min(4, len(annotated))
rows = (len(annotated) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 2.4 * rows))
for ax, im in zip(np.array(axes).ravel(), annotated):
    ax.imshow(im); ax.axis("off")
for ax in np.array(axes).ravel()[len(annotated):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

found = sum(1 for _, a in results if a.visible)
print(f"シャトルが見えたコマ: {found} / {len(results)}")


## 5. 描いたコマをつなげて動画にする

コマ送りの動画にして、点の動きがシャトルの軌道に沿っているかを見ます。


In [ ]:
out_raw = "/content/shuttle_raw.mp4"
out_mp4 = "/content/shuttle_er2.mp4"
W, H = annotated[0].size
writer = cv2.VideoWriter(out_raw, cv2.VideoWriter_fourcc(*"mp4v"), SAMPLE_FPS, (W, H))
for im in annotated:
    writer.write(cv2.cvtColor(np.array(im), cv2.COLOR_RGB2BGR))
writer.release()

# ブラウザで再生できる形式（H.264）に変換
!ffmpeg -y -loglevel error -i {out_raw} -vcodec libx264 -pix_fmt yuv420p -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" {out_mp4}

from base64 import b64encode
data = b64encode(open(out_mp4, "rb").read()).decode()
display(HTML(f'<video width="720" controls loop src="data:video/mp4;base64,{data}"></video>'))
print("保存先:", out_mp4, "（左のファイルペインからダウンロードできます）")


## 6. 動画をまるごと渡して、時刻を聞く（おまけ）

コマではなく動画そのものを Files API でアップロードして、「シャトルが打たれた瞬間は何秒か」を聞きます。
1回の呼び出しで済みますが、位置（座標）は返らず、**時刻**の答えになります。


In [ ]:
print("動画をアップロードしています...")
vid = client.files.upload(file=VIDEO_PATH)
while vid.state.name == "PROCESSING":
    print(".", end="", flush=True); time.sleep(2)
    vid = client.files.get(name=vid.name)
print("\n準備完了:", vid.uri)


class Hit(BaseModel):
    second: float = Field(description="シャトルが打たれた時刻（秒）")
    who: str = Field(description="手前・奥など、打った側の説明（日本語）")


class Hits(BaseModel):
    items: List[Hit] = Field(description="見つかった打球。見つからなければ空のリスト")


response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "video", "uri": vid.uri},
        {"type": "text", "text": "この動画で、シャトルが打たれた瞬間を時刻（秒）つきで全部挙げてください。"
                                 "見えないものは推測しないこと。無ければ空のリストを返してください。"},
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={"type": "text", "mime_type": "application/json", "schema": Hits.model_json_schema()},
)
print(response.output_text)


## 次に試すこと

- `SAMPLE_FPS` を上げる（呼び出し回数が増える。無料枠は 1日 20 リクエスト、有料枠なら 1コマ 1 円弱）
- 外れたコマだけ `thinking_level` を `"medium"` や `"high"` にして聞き直す
- コマの一部（コート半面）を切り出して渡し、シャトルを大きく見せる
- 返ってきた点の列から速度を出す（座標は 0〜1000 の正規化なので、画面サイズを掛けて画素に直す）
